# Markdown RAG Splitting Test Notebook

This notebook shows a practical ingestion pipeline for Markdown RAG:

1. Parse Markdown with `markdown-it-py`
2. Build section-aware chunks from heading hierarchy
3. Apply token splitting only to oversized sections
4. Preview Pinecone-ready records with metadata

It is intentionally self-contained so you can test the chunking strategy before wiring embeddings and Pinecone upserts.


In [ ]:
from __future__ import annotations

import math
import re
from collections import OrderedDict
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from markdown_it import MarkdownIt

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)


In [ ]:
def resolve_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "knowledge_base").exists() and (
            candidate / "pyproject.toml"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the competition project directory from the current working directory."
    )


PROJECT_DIR = resolve_project_dir()
KB_DIR = PROJECT_DIR / "data" / "knowledge_base"

sample_paths = sorted(path.relative_to(KB_DIR.parent) for path in KB_DIR.rglob("*.md"))
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"Markdown files: {len(sample_paths)}")
display(pd.DataFrame({"sample_path": [str(path) for path in sample_paths[:10]]}))


In [ ]:
def normalize_whitespace(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


try:
    import tiktoken
except ImportError:
    tiktoken = None


def estimate_token_count(text: str) -> int:
    cleaned = normalize_whitespace(text)
    if not cleaned:
        return 0
    if tiktoken is not None:
        encoder = tiktoken.get_encoding("cl100k_base")
        return len(encoder.encode(cleaned))
    # Fallback: rough estimate that works acceptably for mixed English/Thai notebook testing.
    return max(1, math.ceil(len(cleaned) / 4))


def format_heading_path(path: list[str]) -> str:
    return " > ".join(path) if path else "(preamble)"


def render_heading_lines(path: list[str]) -> str:
    if not path:
        return ""
    return "\n".join(f"{'#' * (idx + 1)} {heading}" for idx, heading in enumerate(path))


def compose_section_text(heading_path: list[str], block_texts: list[str]) -> str:
    pieces = []
    heading_lines = render_heading_lines(heading_path)
    if heading_lines:
        pieces.append(heading_lines)
    pieces.extend(text for text in block_texts if normalize_whitespace(text))
    return normalize_whitespace("\n\n".join(pieces))


In [ ]:
def parse_markdown_blocks(text: str, source_path: str) -> list[dict[str, Any]]:
    md = MarkdownIt("commonmark").enable("table").enable("strikethrough")
    tokens = md.parse(text)
    lines = text.splitlines()
    stack: list[str | None] = []
    blocks: list[dict[str, Any]] = []

    def current_path() -> list[str]:
        return [item for item in stack if item]

    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token.type == "heading_open":
            level = int(token.tag[1])
            inline_token = tokens[i + 1] if i + 1 < len(tokens) else None
            heading_text = normalize_whitespace(getattr(inline_token, "content", ""))
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = heading_text
            stack[:] = stack[:level]
            i += 1
            continue

        if token.type in {
            "paragraph_open",
            "blockquote_open",
            "bullet_list_open",
            "ordered_list_open",
            "table_open",
        }:
            start_line, end_line = token.map or (None, None)
            segment = (
                "\n".join(lines[start_line:end_line])
                if start_line is not None and end_line is not None
                else ""
            )
            segment = normalize_whitespace(segment)
            if segment:
                blocks.append(
                    {
                        "source_path": source_path,
                        "block_type": token.type.removesuffix("_open"),
                        "heading_path": current_path(),
                        "text": segment,
                    }
                )
        elif token.type == "fence" and token.content:
            fence_text = normalize_whitespace(token.markup + "\n" + token.content)
            blocks.append(
                {
                    "source_path": source_path,
                    "block_type": "fence",
                    "heading_path": current_path(),
                    "text": fence_text,
                }
            )
        i += 1
    return blocks


def build_sections(
    blocks: list[dict[str, Any]], merge_preamble_tokens: int = 80
) -> list[dict[str, Any]]:
    grouped: OrderedDict[tuple[str, ...], list[dict[str, Any]]] = OrderedDict()
    for block in blocks:
        key = tuple(block["heading_path"])
        grouped.setdefault(key, []).append(block)

    sections: list[dict[str, Any]] = []
    for key, grouped_blocks in grouped.items():
        block_texts = [block["text"] for block in grouped_blocks]
        section_text = compose_section_text(list(key), block_texts)
        sections.append(
            {
                "source_path": grouped_blocks[0]["source_path"],
                "heading_path": list(key),
                "block_count": len(grouped_blocks),
                "text": section_text,
                "estimated_tokens": estimate_token_count(section_text),
            }
        )

    if len(sections) >= 2:
        first_section = sections[0]
        second_section = sections[1]
        if (
            len(first_section["heading_path"]) == 1
            and len(second_section["heading_path"]) >= 2
            and second_section["heading_path"][0] == first_section["heading_path"][0]
            and first_section["estimated_tokens"] <= merge_preamble_tokens
        ):
            merged_block_texts = [first_section["text"]]
            child_heading_lines = render_heading_lines(
                second_section["heading_path"][1:]
            )
            if child_heading_lines:
                merged_block_texts.append(child_heading_lines)
            merged_block_texts.append(
                "\n\n".join(
                    block["text"]
                    for block in grouped[tuple(second_section["heading_path"])]
                )
            )
            sections[1] = {
                **second_section,
                "block_count": first_section["block_count"]
                + second_section["block_count"],
                "text": normalize_whitespace(
                    "\n\n".join(part for part in merged_block_texts if part)
                ),
            }
            sections[1]["estimated_tokens"] = estimate_token_count(sections[1]["text"])
            sections = sections[1:]
    return sections


In [ ]:
SOURCE_PATH = "knowledge_base/policies/shipping_policy.md"
raw_text = (KB_DIR.parent / SOURCE_PATH).read_text(encoding="utf-8")

blocks = parse_markdown_blocks(raw_text, SOURCE_PATH)
sections = build_sections(blocks)

sections_df = pd.DataFrame(
    {
        "heading_path": [
            format_heading_path(section["heading_path"]) for section in sections
        ],
        "block_count": [section["block_count"] for section in sections],
        "estimated_tokens": [section["estimated_tokens"] for section in sections],
        "preview": [section["text"][:160] for section in sections],
    }
)

display(sections_df)


## Token splitting oversized sections

The rule here is simple:

- keep a section whole if it already fits the target size
- split only oversized sections
- keep overlap only inside the same section


In [ ]:
def split_text_with_overlap(
    text: str, chunk_size: int = 450, chunk_overlap: int = 75
) -> list[str]:
    text = normalize_whitespace(text)
    if estimate_token_count(text) <= chunk_size:
        return [text]

    separators = ["\n\n", "\n", ". ", " "]
    pieces = [text]
    for separator in separators:
        if all(estimate_token_count(piece) <= chunk_size for piece in pieces):
            break
        next_pieces: list[str] = []
        for piece in pieces:
            if estimate_token_count(piece) <= chunk_size or separator not in piece:
                next_pieces.append(piece)
                continue
            split_parts = [
                normalize_whitespace(part)
                for part in piece.split(separator)
                if normalize_whitespace(part)
            ]
            if separator in {". ", " "}:
                suffix = "." if separator == ". " else ""
                split_parts = [
                    part + suffix if suffix and not part.endswith(suffix) else part
                    for part in split_parts
                ]
            next_pieces.extend(split_parts)
        pieces = next_pieces

    hard_limit = max(200, chunk_size * 4)
    flattened: list[str] = []
    for piece in pieces:
        if estimate_token_count(piece) <= chunk_size:
            flattened.append(piece)
            continue
        for start in range(0, len(piece), hard_limit):
            flattened.append(piece[start : start + hard_limit])

    chunks: list[str] = []
    current = ""
    overlap_chars = max(80, chunk_overlap * 6)
    for piece in flattened:
        piece = normalize_whitespace(piece)
        if not piece:
            continue
        candidate = normalize_whitespace(f"{current}\n\n{piece}" if current else piece)
        if current and estimate_token_count(candidate) > chunk_size:
            chunks.append(current)
            overlap_seed = current[-overlap_chars:]
            current = normalize_whitespace(f"{overlap_seed}\n\n{piece}")
            if estimate_token_count(current) > chunk_size:
                chunks.append(piece)
                current = ""
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


def build_rag_records(
    sections: list[dict[str, Any]], chunk_size: int = 450, chunk_overlap: int = 75
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    next_chunk_number = 0
    for section_index, section in enumerate(sections):
        child_chunks = split_text_with_overlap(
            section["text"], chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )
        for chunk_in_section, chunk_text in enumerate(child_chunks):
            chunk_id = f"{section['source_path']}#chunk_{next_chunk_number:04d}"
            records.append(
                {
                    "id": chunk_id,
                    "chunk_text": chunk_text,
                    "chunk_tokens": estimate_token_count(chunk_text),
                    "metadata": {
                        "document_id": section["source_path"],
                        "file_path": section["source_path"],
                        "heading_path": section["heading_path"],
                        "section_index": section_index,
                        "chunk_number": next_chunk_number,
                        "chunk_in_section": chunk_in_section,
                        "source_type": "markdown",
                    },
                }
            )
            next_chunk_number += 1
    return records


In [ ]:
records = build_rag_records(sections, chunk_size=450, chunk_overlap=75)

records_df = pd.DataFrame(
    {
        "id": [record["id"] for record in records],
        "heading_path": [
            format_heading_path(record["metadata"]["heading_path"])
            for record in records
        ],
        "chunk_tokens": [record["chunk_tokens"] for record in records],
        "preview": [record["chunk_text"][:180] for record in records],
    }
)

display(records_df)
print(f"Sections: {len(sections)}")
print(f"Final Pinecone records: {len(records)}")


In [ ]:
records[:2]


## How to use this notebook

- Change `SOURCE_PATH` to another markdown file in `data/knowledge_base`
- Tune `chunk_size` and `chunk_overlap`
- Once the chunk shape looks right, replace the fallback token estimator with `tiktoken` if you add it later
- After that, embed `record["chunk_text"]` and upsert `id + metadata` into Pinecone
